# Data loading - Eurostat

In [23]:
import requests
import gzip
import pandas as pd
import io
from pathlib import Path

In [ ]:
# Eurostat's SDMX 2.1 bulk download
# {code} is filled in per dataset; compressed=true means the response body is gzip-encoded TSV.

BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{code}?format=TSV&compressed=true"

# Dataset codes to download, mapped to a human-readable label (for your own reference,
# not used by the code below).
eurostat_datasets={
    'nama_10r_2gdp': 'GDP',
    'lfst_r_lfu3rt': 'unemployment_rate',
    'isoc_r_iuse_i': 'internet_usage',
    'ilc_li41': 'poverty_risk',
    'tps00208': 'life_expectancy',
    'tps00100': 'social_expenditure',
    'hlth_ehis_fv3e': 'veggie_consumption',
    'hlth_ehis_pe2e': 'aerobic_activity',
    'hlth_ehis_al3e': 'heavy_drinking',
    'hlth_ehis_sk3e': 'smoking',
    'sdg_02_10': 'ubesity_rate',
    'ilc_pw01b': 'life_satisfaction',
    'ilc_scp09': 'family_contact',
    'ilc_scp19': 'voluntary_participation',
    'ilc_scp15': 'social_support',
    'ilc_scp17': 'social_contact',
    'ilc_pw03': 'social_trust',
}

RAW_DIR = 'data/raw'


def download_and_save(code):
    """
    Download a single Eurostat dataset by code and save it as a plain-text
    .tsv file in RAW_DIR, named after its human-readable label rather than
    its raw code. Skips the download if the file already exists.
    """
    save_path = RAW_DIR / f'{eurostat_datasets[code]}.tsv'

    # Don't re-download files that are already on disk.
    if save_path.exists():
        print(f'SKIP {eurostat_datasets[code]} -> already exists')
        return save_path

    # Request URL for dataset code and fetch it.
    url = BASE_URL.format(code=code)
    r = requests.get(url, timeout=120)
    r.raise_for_status() # raises an exception on HTTP error status codes (404, 500, etc.)

    # Checking to have gzip data, not HTML
    content = r.content
    if content[:2] != b"\x1f\x8b":  # gzip magic number
        raise ValueError(f"[{code}] response is not gzip data (got {content[:30]!r})")

    with gzip.open(io.BytesIO(r.content), "rt", encoding="utf-8") as f:
        df = pd.read_csv(f, sep="\t", na_values=[":", ": "])
    return df

In [22]:
df = download('nama_10r_2gdp')
df.head()

,"freq,unit,geo\TIME_PERIOD",2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,"A,EUR_HAB,AL",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3100,...,3600,3800,4100,4500,4900,4700,5400,6500,NaN,NaN
1,"A,EUR_HAB,AL0",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3100,...,3600,3800,4100,4500,4900,4700,5400,6500,NaN,NaN
2,"A,EUR_HAB,AL01",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2400,...,2900,3100,3300,3600,3900,3600,4300,5300,NaN,NaN
3,"A,EUR_HAB,AL02",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4200,...,4300,4500,5000,5500,5900,5800,6500,7700,NaN,NaN
4,"A,EUR_HAB,AL03",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2500,...,3300,3500,3700,4200,4500,4200,5000,6000,NaN,NaN
